In [ ]:
import json
from openai import OpenAI

with open('configuration.json', 'r') as f:
    conf = json.load(f)
    api_key = conf['openai_key']

client = OpenAI(api_key=api_key)

## ***영상 리스트***
- 프로젝트 초기에 각자 수집했던 동영상 링크(org_video_link.csv)
- 이 중에서 사기 영상 데이터로 사용하기에 부적합한 10개는 제외하여 최종적으로 filtered_video_link.csv 사용

In [ ]:
import pandas as pd

df = pd.read_csv('filtered_video_link.csv')
df

## ***영상 다운로드***
- yt_dlp로 다운로드
- 위의 각 영상 라벨에 따른 디렉토리에 저장
- 파일명은 위 각 영상별 리스트 내의 인덱스번호를 따름
  - 영상 제목에 각종 유니코드 특수문자가 있을 경우 파일명을 정상적으로 인식하지 못하는 경우가 있기 때문
  - 영상의 번호에 따른 원본 추적은 위의 인덱스를 통해 찾으면 됨

In [ ]:
import yt_dlp
import os
import traceback
from pytz import timezone
from datetime import datetime

download_dir = datetime.now(timezone('Asia/Seoul')).strftime('data-%Y-%m-%d')
os.makedirs(download_dir, exist_ok=True)

for i, d in enumerate(df.iterrows()):
    label = d[1]['label']
    url = d[1]['link']
    
    try:
        target_dir = os.path.join(download_dir, label)
        os.makedirs(target_dir, exist_ok=True)

        file_name =  f'{i:04d}.mp4'
        file_path = os.path.join(target_dir, file_name)

        ydl_opts = {
            "outtmpl": file_path,   # 저장 파일 이름: "제목.확장자"
            "format": "mp4",                  # mp4 형식으로 다운로드
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
    except:
        traceback.print_exc()


### ***영상 하나에 대한 설명 추출하기***
- openai API를 이용해 동영상을 인식시키는 방법은 아래와 같음
  - https://cookbook.openai.com/examples/gpt_with_vision_for_video_understanding
  - 결국 프레임 추출하여 이미지를 인식시키는 것
  - 이미지의 인코딩된 스트림이 들어가는 것은 YOLO와 차이가 있음

In [ ]:
import cv2
import base64
import cv2
import numpy as np
from openai import OpenAI

base_dir = os.path.join(download_dir, 'abnormal')
video_descriptions = {}
abnormal_video_descriptions = []

for root, dirs, files in os.walk(base_dir):
    for f in files:
        file_path = os.path.join(root, f)
        video = cv2.VideoCapture(file_path)

        base64Frames = []
        while video.isOpened():
            success, frame = video.read()
            if not success:
                break
            _, buffer = cv2.imencode(".jpg", frame)
            base64Frames.append(base64.b64encode(buffer).decode("utf-8"))

        video.release()
        sample_frames = [base64Frames[i] for i in np.linspace(0, len(base64Frames) - 1, 10, dtype=int)]
        print(f"{file_path}: {len(base64Frames)} frames read with {len(sample_frames)} samples")        

        response = client.responses.create(
            model="gpt-4o-mini",
            input=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "input_text",
                            "text": (
                                """이 비디오 프레임은 일반적인 광고등을 나타내는 영상입니다
                                사기의도를 가진 영상은 아닙니다
        이를 베이스로하여 영상에서 무슨일이 일어나고 있는지 설명하세요
        """
                            )
                        },
                        *[
                            {
                                "type": "input_image",
                                "image_url": f"data:image/jpeg;base64,{frame}"
                            }
                            
                            for frame in sample_frames
                        ]
                    ]
                }
            ],
        )

        print(response.output_text)
        abnormal_video_descriptions.append({"filename": file_path, "description": response.output_text})

In [ ]:
"""[{'filename': '/home/ubuntu/video_20251104/abnormal/0017.mp4',
  'description': '영상은 "테라피 소통방"이라고 소개된 채팅방의 메시지를 보여주고 있습니다. 메시지의 내용은 다음과 같은 요소들이 포함되어 있습니다:\n\n1. **소통방의 목적**: 정보 교류와 피드백을 위한 공간이라는 설명.\n2. **규칙 안내**: 무단 홍보, 특정 링크 공유, 패드립 등 금지 사항이 나열되어 있습니다. 이러한 규칙은 스팸과 부적절한 내용의 유입을 방지하려는 의도로 보입니다.\n3. **긴급성 및 희소성**: “지금” 또는 “한정”이라는 직접적인 언급은 없지만, 규칙을 강조함으로써 참여의 중요성을 환기시키는 방식으로 긴급성을 유도할 수 있습니다.\n\n영상의 전반적인 흐름은 사용자를 특정 소통방에 유도하고, 따르도록 독려하는 구조로 보입니다. 이러한 구조는 사기성 요소를 내포할 수 있습니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0032.mp4',
  'description': '이 영상의 내용은 사기 의도와 관련된 것으로 보입니다. 다음과 같은 요소들이 있습니다:\n\n1. **금전 요구**: 처음에 언급된 요구사항들이 사전 비용(운영비, 수수료 등)과 관련된 것으로 보이며, 이는 피해자에게 금전적 부담을 줄 수 있는 방식입니다.\n\n2. **긴급성·희소성**: "지금", "한정" 같은 긴급한 표현이 사용되면 사람들을 서두르게 하고 더 많은 금전적 결정을 하도록 유도할 수 있습니다.\n\n3. **외부 채널 유도**: QR 코드나 텔레그램과 같은 외부 채널로 연결하려는 시도가 보입니다. 이는 사기행위와 관련된 의사소통을 비공식적으로 하도록 유도하는 방식입니다.\n\n4. **기관 사칭**: 신뢰를 주기 위해 기관이나 언론의 로고를 사용하거나 사칭할 수 있습니다.\n\n5. **과장된 수익 약속**: "위탁은 수익을"과 같은 표현을 통해 무위험 약속이나 과장된 수익을 제시하고 있습니다.\n\n6. **최소비용이 적시됨**: 최소 비용이나 초기 투자금액을 제시하여 사람들을 유인하고, 이후 비용이 점차 증가하는 방식입니다.\n\n7. **사항 과시**: "사람이 필요하다"는 부분은 신뢰를 주기 위한 말로, 실질적으로는 조작된 가짜 직원이거나 사라지는 방식일 수 있습니다.\n\n이 모든 요소들이 결합되어 사람들에게 금전적 손실을 초래할 수 있는 잠재적으로 위험한 상황을 만들어 내고 있습니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0026.mp4',
  'description': "이 비디오는 사기 전화의 전형적인 모습으로 보입니다. 추출된 프레임을 통해 다음과 같은 내용을 확인할 수 있습니다:\n\n1. **보이스피싱 형식**: 대화의 주된 주체는 사기꾼으로 보이며, 검찰이나 경찰을 사칭하고 있습니다. 이는 기관 사칭의 요소가 명백하게 드러나 있습니다.\n\n2. **위험한 상황 강조**: '당신의 계좌가 범죄에 연루되었다'는 식의 긴급한 메시지를 전달하여 상대방의 긴장감을 유도하고 있습니다. 이는 긴급성과 희소성의 요소를 포함합니다.\n\n3. **구체적인 지시**: 사기꾼이 피해자에게 특정 은행 계좌로 돈을 송금하도록 유도하고, 이를 통해 피해자가 직접적인 금전 요구를 받게 됩니다.\n\n4. **법적 조치 연결**: 영상에서 언급된 내용은 경찰 및 법적 절차와의 연관성을 부각시키며, 피해자가 소극적으로 반응하게 만들려는 의도를 보입니다.\n\n5. **전문적인 이미지**: '변호사'라는 신분을 강조하여 사람들에게 신뢰감을 주려고 하며, 이는 사기의 일반적인 기법입니다.\n\n이 비디오는 보이스피싱 사기의 전형적인 사례로, 시민들에게 경각심을 일깨우기 위한 교육적 목적으로도 활용될 수 있습니다."},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0008.mp4',
  'description': '죄송하지만, 요청하신 내용을 처리할 수 없습니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0030.mp4',
  'description': '이 영상은 사기 및 사기의도를 보여주는 내용으로 보입니다. 아래의 카테고리를 기준으로 설명하자면:\n\n1. **금전 요구**: 영상에서 아이디어나 사업 모델을 설명하며 금전적 투자를 요구할 가능성이 있습니다.\n2. **긴급성·희소성**: 사람들에게 즉각적으로 반응하도록 유도하는 언급이 있을 수 있습니다 ("지금", "한정").\n3. **외부 채널 유도**: 대화 중에 특정 플랫폼(예: 텔레그램, QR 코드 등)으로의 전환을 유도하는 내용이 있을 수 있습니다.\n4. **기관/언론/은행 사칭 정황**: 신뢰성을 높이기 위해 유명 브랜드나 기관의 로고를 사용하고 있을 수 있습니다.\n5. **과장 수익/무위험 약속**: 높은 수익을 보장하거나 무위험으로 자본을 투자하라는 메시지가 포함될 수 있습니다.\n\n이 모든 요소들이 모여서 참여자들에게 믿음을 주고 그들의 투자를 유도하는 방식으로 작용하고 있음을 보입니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0002.mp4',
  'description': '이 영상에서 나타나는 내용은 여러 가지 사기적인 요소를 포함하고 있습니다. 다음은 주요 포인트입니다:\n\n1. **과장된 수익 주장**: "수백 배 가능"과 같은 문구가 나타나며 조직 우위의 수익을 나열하고 있습니다. 이는 과장된 투자 수익을 약속하여 사람들을 유혹하는 방식입니다.\n\n2. **긴급성 및 제한성 강조**: 영상은 "지금"이나 "한정"이라는 단어를 사용하여 즉각적인 행동을 유도하고 있습니다.\n\n3. **불법적인 베팅 또는 투자의지**: 합법적이라고 주장하면서도 고위험적인 베팅을 권장하고 있고, 매주 일정 금액까지 베팅이 가능하다는 내용을 담고 있습니다.\n\n4. **해외 채널 유도**: 이러한 웹사이트나 플랫폼으로 유도하고 있으며, 신뢰성을 높이기 위해 특정 로고나 이미지 사용 가능성이 있습니다.\n\n5. **과장된 이미지와 성과 통계**: 수익률이나 승률을 비현실적으로 높게 설정하여 마케팅하는 모습이 보입니다.\n\n6. **가짜 정보 제공**: 스포츠 분석이나 과학적 데이터에 기반한 것처럼 꾸며진 콘텐츠를 통해 신뢰성을 가장하고 있습니다.\n\n결론적으로, 영상을 통해 은밀하게 사람들을 속이려는 시도가 드러나며, 이러한 요소들은 모두 사기의 전형적인 징후로 볼 수 있습니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0037.mp4',
  'description': '이 영상은 사기 관련 콘텐츠로 보이며, 여러 사기의 요소들을 포함하고 있습니다. 프레임에서는 주식 투자에 관련된 내용이 다뤄지고 있고, 그 과정에서 다음과 같은 특징들이 나타납니다:\n\n1. **금전 요구**: 특정 가격(1,225원에서 14,000원으로의 상승)을 목표로 하는 주식이 소개되고 있습니다.\n2. **긴급성**: "지금"과 같은 긴급한 표현이 사용되어 투자 결정을 서두르게 유도합니다.\n3. **외부 채널 유도**: LINE ID와 같은 정보를 제시하여 개인적인 연결을 유도합니다.\n4. **과장된 수익 약속**: 주식의 상승 가능성을 과장하여 제시합니다.\n5. **연락처 노출**: 개인 연락처가 화면에 드러납니다.\n\n이러한 요소들은 일반적으로 사기성 투자 권유에서 발견되는 전형적인 패턴이며, 많은 주의가 필요합니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0031.mp4',
  'description': "이 영상에서 일어나고 있는 일은 특정 앱을 통해 사용자가 활동하는 모습으로 보입니다. 화면에는 'ADLIX'라는 프로그램의 인터페이스가 나타나 있으며, 사용자는 개인 계정을 관리하고 있는 것으로 추정됩니다.\n\n1. **금전 요구**: 10,000이라는 금액이 표시되어 있어 이 프로그램이 사용자의 이익을 요구하고 있을 가능성이 있습니다.\n2. **긴급성·희소성**: 사용자가 이 시스템에 대해 빠르게 반응할 필요성이 있는지 여부는 명확하지 않지만, 일반적으로 이런 형태의 앱에서는 긴급한 요청이 있을 수 있습니다.\n3. **외부 채널 유도**: 특정 채널 등록이나 커뮤니티 활동을 유도하는 메뉴들이 보여, 사용자가 소셜 미디어와 연결될 가능성이 있습니다.\n4. **과장 수익**: 앱 사용의 목적과 수익 모델에 대해 과장된 약속이 존재할 수 있습니다.\n5. **연락처 및 URL 노출**: 화면에 다양한 기능이 있어, 사용자가 개인 정보를 노출할 여지가 있습니다.\n\n전반적으로 이 영상은 잠재적인 사기 시나리오와 관련이 있을 수 있으며, 사용자는 주의가 필요합니다."},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0014.mp4',
  'description': '영상은 최근 유행하는 신종 피싱을 다루고 있는 것으로 보입니다. 주 내용은 한 남자가 다른 남자에게 신종 사기 방식에 대해 이야기하는 형식으로 진행됩니다. \n\n1. **금전 요구**: 영상에서는 수수료나 선입금 같은 금전적인 요구가 언급될 가능성이 있습니다.\n2. **긴급성·희소성**: 특정 순간이나 기회가 놓쳐지면 안 된다는 점을 강조하며 긴박하게 느끼게 할 수 있습니다.\n3. **외부 채널 유도**: 개인정보나 특정 앱으로의 이동을 유도하는 방식이 사용될 수 있습니다.\n4. **기관 사칭**: 피싱범이 마치 신뢰할 수 있는 기관처럼 행세할 가능성이 높습니다.\n5. **과장 수익**: 피싱범이 수익이 높을 것이라고 약속할 수 있으며, 이는 사기 수법에 일반적으로 사용됩니다.\n\n영상의 전반적인 흐름은 피싱 사기 방식과 그에 대한 경각심을 일깨우기 위한 메시지를 전달하는 것처럼 보입니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0007.mp4',
  'description': "이 영상은 사이비 교주가 사람들에게 꿈과 희망을 주겠다는 메시지를 전달하는 장면으로 보입니다. 교주는 정장 차림으로, 여러 사람들과 상호작용하며 신뢰를 구축하려고 하는 것 같습니다. 영상 속 인물들은 모두 마스크를 착용하고 있으며, 교주가 특정 인물에게 다가가 손을 대거나 물리적으로 어떤 행동을 취하는 장면들이 반복적으로 나타납니다. \n\n이 과정에서 교주는 신비로운 방식으로 '희망'이나 '변화'를 전달하려고 하는 모습이 보이며, 주위 사람들은 그에 집중하고 있는 듯한 표정을 짓고 있습니다. 이러한 대화나 제스처는 일반적으로 사람들을 끌어들이거나 흥미를 유도하는 역할을 합니다. \n\n이와 관련하여 앞서 명시된 사기적 요소와 연관지어 생각해보면, 교주가 사람들에게 금전적 요구나 특정 행동을 하도록 유도할 가능성이 있음을 염두에 두어야 하겠습니다. 전반적으로, 이 영상은 사기와 관련된 신빙성 없는 메시지를 전달하려는 의도를 가진 것으로 보입니다."},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0018.mp4',
  'description': '이 영상은 특정 소셜 미디어 플랫폼에서의 채팅 화면으로 보이며, 여러 가지 사기성 요소를 포함하고 있을 가능성이 있습니다. 여기서 분석할 수 있는 몇 가지 요소는 다음과 같습니다.\n\n1. **외부 채널 유도**: 영상 내 대화에서 특정 링크나 채널로의 유도는 나타나지 않지만, 그런 내용이 있을 수 있는 문맥을 고려해야 합니다.\n\n2. **긴급성 및 희소성**: "다들 모여라"는 문구가 나타나며, 이는 사용자들에게 신속한 행동을 유도하려는 전략일 수 있습니다.\n\n3. **금전 요구**: 대화의 구체적인 내용을 알 수는 없지만, 채팅 내용에서 금전적인 요구가 있을 가능성이 존재합니다.\n\n4. **과장 수익/무위험 약속**: 특정 사용자들이 수익에 관련된 내용을 언급하면, 이는 사기임을 암시할 수 있습니다.\n\n5. **특정 이미지 사용**: 영상 속 인물의 사진이 사기와 관련하여 매력이나 신뢰성을 주기 위한 수단으로 사용될 수도 있습니다.\n\n전체적으로 이 영상은 소셜 미디어 플랫폼을 이용한 사기 행위의 일부로 보이며, 사용자는 신중하게 행동해야 할 필요가 있습니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0005.mp4',
  'description': '해당 영상 프레임들은 "가상화폐 구매 대행 모집"이라는 내용을 강조하고 있습니다. 주요 포인트는 다음과 같습니다:\n\n1. **금전 요구**: "한 달이면 마음에 드는 차를 살 수 있습니다"라는 문구는 사용자에게 낮은 비용으로 큰 혜택을 제안하는 형태입니다.\n   \n2. **긴급성·희소성**: 특정 기간 내에 참여해야 하는 느낌을 주어, 서두르도록 유도합니다.\n\n3. **고정 배너/전화번호 오버레이**: 직접적인 연락처나 추가 정보를 요구하는 형태로 보입니다.\n\n이런 요소들을 보아, 가상화폐와 관련된 투자 사기나 불법적인 금융 거래를 암시하는 것일 수 있습니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0016.mp4',
  'description': '이 영상은 사기 의도가 담긴 콘텐츠로 보입니다. 다음과 같은 특징이 있습니다:\n\n1. **금전 요구**: 화면에 표시된 "보유머니 2,500,000"이라는 문구는 사용자가 일정 금액을 보유하고 있다는 사실을 강조하여 돈을 요구하는 목적으로 사용될 수 있습니다.\n\n2. **과장된 수익 약속**: 높은 수치가 강조되어 있어 잠재적 투자자들에게 과장된 이익을 예고하는 메시지를 전달합니다.\n\n3. **긴급성 및 희소성**: "지금"과 같은 단어들이 사용될 가능성이 높으며, 이를 통해 사용자는 즉각적인 반응을 유도받을 수 있습니다.\n\n4. **외부 채널 유도**: 사용자가 추가 정보를 위해 다른 플랫폼으로 이동하게 만드는 유도적 요소가 포함될 수 있습니다.\n\n이러한 요소들은 사용자로 하여금 사기 피해를 입게 할 위험이 크므로 주의가 필요합니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0006.mp4',
  'description': '이 비디오에서는 연속적으로 여러 장의 돈다발을 보여주며, "디지털 화폐의 고가 매입 온라인 아르바이트 모집. 하루에 50만 원씩 벌어요"라는 메시지가 반복적으로 나타납니다. 이 문구는 과장된 수익 약속을 내포하고 있으며, 사기와 관련된 요소를 가지고 있습니다.\n\n주요 요소들은 다음과 같습니다:\n\n1. **과장된 수익 약속**: 하루에 50만 원을 벌 수 있다는 주장은 신뢰를 주기 위한 과장된 메시지입니다.\n2. **긴급성 유도**: \'온라인 아르바이트 모집\'이라고 하여, 즉각적인 반응을 유도하는 분위기를 조성합니다.\n3. **금전 요구의 가능성**: 초기 비용이나 선입금을 요구할 가능성이 크며, 사용자를 유도하는 형태를 띱니다.\n\n이러한 요소들은 사기 성격의 광고로 의심스러우며, 일반적으로 합법적이고 안전한 투자 기회에서는 느껴지지 않는 압박과 긴급성이 특징입니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0001.mp4',
  'description': '이 영상의 연속 프레임은 사기 수법을 보여주는 것 같습니다. 다음과 같은 요소들이 포함되어 있습니다:\n\n1. **금전 요구**: "카톡문의의 aia771" 및 "텔레그램 aia771"과 같은 문구는 금전을 요구하거나 특정 거래를 유도하고 있음을 시사합니다.\n\n2. **긴급성·희소성**: "오후 나스닥 인증"과 같은 글귀는 급박감을 조성하여 빠른 결정을 유도하고 있습니다.\n\n3. **외부 채널 유도**: 특정 카카오톡이나 텔레그램 링크를 통해 개인 대화로 유도하고 있으며, 일반적으로 이런 방식은 사기와 관련이 있습니다.\n\n4. **과장된 이익**: 차트를 통한 투자 수익을 과장하여 보여주는 것으로, 무위험 투자 약속처럼 보이게 하고 있습니다.\n\n5. **고정 배너/전화번호 오버레이**: 반복적으로 나타나는 "카톡문의", "텔레그램" 문구는 사람들에게 특정 연락방법으로 접근하도록 하려는 의도가 있습니다.\n\n전반적으로 이 영상은 투자 사기와 관련된 내용으로 해석될 수 있으며, 피해를 입지 않도록 주의가 필요합니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0035.mp4',
  'description': '이 비디오는 주식 거래 또는 암호화폐 투자와 관련된 사기 광고를 담고 있는 것으로 보입니다. 다음과 같은 요소들이 그 징후를 보여줍니다:\n\n1. **급박한 신호**: "10월 전에는 꼭 사라"라는 메시지는 긴급한 행동을 유도하여 투자자들이 즉시 결정을 내리도록 압박합니다.\n\n2. **과장된 이익**: 높은 수익률을 강조하며, "무위험" 이라는 표현은 투자에 대한 과장된 약속으로 해석될 수 있습니다.\n\n3. **거래 내용 노출**: 특정 주식의 가격과 거래 정보를 반복적으로 보여주어 신뢰성을 높이려는 시도가 있습니다.\n\n4. **외부 플랫폼 유도**: 외부 URL이나 앱으로의 유도는 가짜 거래 플랫폼으로 투자자를 유인할 수 있습니다.\n\n5. **로고 및 기관 사칭**: 특정 기업명이나 로고를 사용하여 신뢰성을 높이려는 모습이 담길 수 있습니다.\n\n이 영상은 투자에 대한 조작된 정보를 통해 사용자를 유인하려는 사기성 콘텐츠로 판단됩니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0033.mp4',
  'description': "이 비디오 프레임들은 특정 금융 사기의 형태인 '돼지 도살 사기'를 다루고 있습니다. 이 사기는 사기 피해자에게 투자 수익을 약속하면서 금전을 요구하는 방식으로 진행됩니다. \n\n### 비디오의 주요 내용:\n1. **전화 통화**: 첫 번째 프레임에서는 의심스러운 전화 통화가 이루어지고 있으며, 이는 사기꾼이 금전을 필요로 하는 상황을 제시하고 있습니다.\n  \n2. **뉴스 보도**: 두 번째 프레임에서는 사기 사건에 대해 보도하는 뉴스 기사가 보여주고, 일본 투자 관련 내용을 언급합니다.\n\n3. **허위 증거**: 세 번째 프레임에서는 유명인사의 여권과 같은 문서를 보여주며 신뢰성을 높이는 장치를 사용하고 있습니다. 이는 사기 피해자가 보다 쉽게 믿을 수 있도록 유도합니다.\n\n4. **영상 통화**: 네 번째 프레임에서는 사기범이 피해자와의 화상 통화에서 자신을 속여 신뢰를 주려는 상황을 보여줍니다.\n\n5. **메신저 대화**: 다섯 번째 프레임은 남는 수익을 보장하고 적은 금액으로 시작할 것을 독려하는 메시지를 보여줍니다.\n\n6. **전문가의 경고**: 마지막 프레임들에서는 전문가들이 이러한 사기의 위험성에 대해 경고하고, 사기를 예방하기 위한 방법을 설명합니다.\n\n### 전체 흐름:\n이 영상에서는 사기범이 어떻게 신뢰를 구축하고 피해자를 끌어들여 금전을 요구하는지를 보여주고 있습니다. 또한, 피해를 예방할 수 있는 정보와 공공의 인식을 높이려는 노력이 담겨 있습니다."},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0029.mp4',
  'description': '이 영상은 분명히 사기성 투자 제안의 특성을 띠고 있습니다. 다음과 같은 요소들이 존재합니다:\n\n1. **금전 요구**: "100% 보장된 나무 코인 수확 시작"이라는 문구가 있어, 투자자에게 금전적 투자를 유도합니다.\n\n2. **긴급성·희소성**: 특정 기간(2025년 8월 1일부터 9월 30일까지) 동안만 혜택이 제공된다고 하여, 빠른 결정을 요구하고 있습니다.\n\n3. **과장 수익/무위험 약속**: "확률 NO!"와 같은 표현을 통해 위험이 없다고 강조하며, 수익을 과장하여 표시하고 있습니다.\n\n4. **고정 배너 및 전화번호 오버레이**: 자주 사용되는 슬로건 및 특정 제품 배너가 화면에 나타나, 공식 광고처럼 보이도록 합니다.\n\n5. **투자/수익·거래 내역 캡쳐 남발**: 지속적으로 수익을 강조하며 잠재적 투자자들에게 유혹합니다.\n\n6. **외부 채널 유도**: 이러한 문구는 일반적으로 비즈니스 또는 투자에 대해 신뢰성을 높이고 의사 결정을 유도하기 위해 사용됩니다.\n\n이 모든 요소는 사용자에게 믿음을 주고, 결국 투자금을 유도하는 방향으로 설계된 전형적인 사기성 콘텐츠입니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0034.mp4',
  'description': '이 비디오 프레임에서 나타나는 내용은 투자 관련 사기의 요소를 포함하고 있습니다. 다음과 같은 카테고리에 해당합니다:\n\n1. **금전 요구:** 현재 주가와 목표 주가를 설정하여 투자 유도를 하고 있으며, 투자 금액과 예상 수익까지 제시하고 있습니다.\n2. **긴급성·희소성:** "현재"라는 단어를 사용하여 빠른 결정을 요구하고 있습니다.\n3. **과장 수익/무위험 약속:** 1054%의 수익률을 강조하며, 무위험으로 보이는 투자 성과를 주장하고 있습니다.\n4. **기관/언론/은행 사칭:** JP모건체이스 같은 유명한 금융 기관을 언급하여 신뢰성을 높이고 있습니다.\n5. **연락처 노출:** 특정 수치나 링크를 통해 직접적인 연락을 유도할 가능성이 있습니다.\n\n비디오에서 제시된 정보는 투자자들에게 불리한 결과를 초래할 수 있는 사기의 징후가 명백합니다. 각 요소는 신중하게 검토되어야 하며, 합법적인 투자 기회인지 철저한 조사가 필요합니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0011.mp4',
  'description': '이 이미지들은 온라인 사기 시나리오를 반영하고 있는 것으로 보입니다. 전반적으로 이러한 프레임들은 두 사람 간의 대화를 담고 있으며, 주제로는 금융 사기, 신원 사칭, 긴급성 부각 등이 포함될 수 있습니다.\n\n1. **신원과 진실성**: 한 개인이 가면을 쓰고 등장해 신원을 숨기고 있으며, 이는 사기꾼의 특징처럼 보입니다.\n2. **제안과 요구**: 대화에서 특정 정보를 요구하거나, 이메일 주소나 은행 계좌 번호 같은 민감한 정보를 요청하는 상황이 나타나 있습니다.\n3. **긴급성**: “지금”이라는 단어가 강조되며, 이는 사람들에게 즉각적인 행동을 유도하는 전형적인 사기 기법입니다.\n4. **과장된 주장**: 대화에서 자주 사용되는 있는 과장된 언어는 투자와 관련된 허위 정보로 추측됩니다.\n5. **유머와 조롱**: 프레임의 일부는 상황을 조롱하거나 웃기게 표현하고 있으며, 이는 사람들로 하여금 심각함을 잊게 만들 수 있습니다.\n\n이러한 요소들은 사기꾼들이 사람들을 속이기 위해 사용하는 일반적인 전략들로, 경계가 필요한 상황임을 상기시킵니다.'},
 {'filename': '/home/ubuntu/video_20251104/abnormal/0025.mp4',
  'description': '죄송하지만, 그 요청을 도와드릴 수 없습니다.'}]
"""

abnormal_video_descriptions

### ***설명이 이상한것 다시 테스트***
- 이를 위해 단건 영상을 설명하기 위한 explain_video() 함수를 따로 뺌

In [ ]:
import cv2
import base64
import cv2
import numpy as np
from openai import OpenAI

def explain_video(file_path, prompt):
    video = cv2.VideoCapture(file_path)

    base64Frames = []
    while video.isOpened():
        success, frame = video.read()
        if not success:
            break
        _, buffer = cv2.imencode(".jpg", frame)
        base64Frames.append(base64.b64encode(buffer).decode("utf-8"))

    video.release()
    sample_frames = [base64Frames[i] for i in np.linspace(0, len(base64Frames) - 1, 10, dtype=int)]
    print(f"{file_path}: {len(base64Frames)} frames read with {len(sample_frames)} samples")        

    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": (prompt)
                    },
                    *[
                        {
                            "type": "input_image",
                            "image_url": f"data:image/jpeg;base64,{frame}"
                        }
                        
                        for frame in sample_frames
                    ]
                ]
            }
        ],
    )

    return response.output_text

### ***추가 시도***

In [ ]:
import os

video_path = os.path.join(download_dir, 'abnormal', '0025.mp4')
prompt = """이 비디오 프레임들은 하나의 비디오로부터 추출된 연속 프레임 입니다
    이 영상은 사기의도를 담고 있는 영상으로 아래와 같은 카테고리중 하나입니다
    -금전 요구(선입금/수수료/상품권/코인 전송)
    -긴급성·희소성(“지금”, “한정”)
    -외부 채널 유도(오픈채팅/텔레그램/QR)
    -기관/언론/은행 사칭 정황(로고, 허가증 이미지)
    -과장 수익/무위험 약속
    -고정 배너/전화번호 오버레이
    -투자/수익·거래 내역 캡쳐 남발
    -연락처·지갑주소·짧은 URL 노출

    이를 베이스로하여 영상에서 무슨일이 일어나고 있는지 설명하세요
    """

explain_video(video_path, prompt)

In [ ]:
abnormal_video_descriptions.pop(-1)

In [ ]:
abnormal_video_descriptions.append(
    {
        "filename": "/home/ubuntu/video_20251104/abnormal/0025.mp4",
        "description": '비디오 프레임을 바탕으로 설명하겠습니다. 이 영상은 다음과 같이 분류할 수 있는 요소들을 포함하고 있을 수 있습니다:\n\n1. **금전 요구**: 전화기로 보여주는 화면이나 언급된 사항이 선입금 요구나 수수료에 관한 내용일 수 있습니다.\n   \n2. **긴급성·희소성**: "지금"과 같은 긴급한 느낌을 주는 언어 사용이 있을 수 있습니다.\n\n3. **외부 채널 유도**: 특정한 채널이나 소셜 미디어 플랫폼으로의 유도가 이루어질 가능성이 있습니다.\n\n4. **기관 사칭 정황**: 특정 로고나 공식 문서의 이미지를 사용하여 신뢰성을 높이려는 노력이 있을 수 있습니다.\n\n5. **과장 수익/무위험 약속**: 투자에 대한 높은 수익률을 약속하는 내용이 포함될 수도 있습니다.\n\n프레임 속 인물의 표정이나 몸짓이 저 혼란스러움이나 불확실성을 나타내는 것처럼 보인다면, 신뢰성을 떨어뜨릴 수 있는 요소로 작용할 수 있습니다. 이 모든 요소는 사전에 설정된 전략과 함께 사기의도를 효과적으로 드러낼 수 있는 방법으로 사용될 수 있습니다. \n\n따라서 이 영상에서는 사기와 관련된 언급이나 요소들이 포함되어 있을 가능성이 높습니다.'
    }
)

In [ ]:
abnormal_video_descriptions

## ***추가해야할 프롬프트***
- normal_video_descriptions 를 추가 필요

In [ ]:
# Please complete this
normal_video_descriptions = []

In [ ]:
import os


prompt = f"""이제부터 MiniCPM-o-2_6에게 지시할 프롬프트를 만들겠습니다
주어진 영상이 사기에 해당하는지 아닌지 판별하는 영상입니다
우리는 {len(abnormal_video_descriptions)}개의 비디오로부터 아래와 같은 요약설명을 추출하였습니다
"""

for abnormal_video_description in abnormal_video_descriptions:
    filename = abnormal_video_description['filename']
    description = abnormal_video_description['description']

    prompt+=f'{os.path.basename(filename)}에서는 다음과 같은 요약설명이 있었습니다\n{description}\n'


prompt+=f"""한편, 정상적인 사기의도가 없는 영상은 {len(normal_video_descriptions)}개의 비디오로부터 아래와 같은 요약설명을 추출하였습니다"""
for normal_video_description in normal_video_descriptions:
    filename = abnormal_video_description['filename']
    description = abnormal_video_description['description']

    prompt+=f'{os.path.basename(filename)}에서는 다음과 같은 요약설명이 있었습니다\n{description}\n'


prompt+="""=========================
여기까지 입니다
이러한 사기에 해당하는 요약설명이 추출될 수 있다면 그것은 사기영상 입니다
그러나 정상적인 사기의도가 없는 요약설명이 추출될 수 있다면 그것은 사기영상이 아닌 정상 입니다
이렇게 영상에 사기 징후가 있는지 없는지 탐지할 수 있도록 MiniCPM에게 영어로 프롬프트를 만들어주세요
MiniCPM은 아래와 같이 출력할 수 있어야 합니다
   {
     "is_scam": true | false,
     "confidence": 0.0 ~ 1.0,
     "risk" : one of "low, mid, high"
     "evidence": ["blabla1", "blabla2"],
     "explanation": "blablablabla1blabla1blabla1blabla1blabla"
   }
"""
print(prompt)

### ***Chat GPT API에서 아래와 같이 프롬프트를 얻음***
- 프롬프트가 만족스럽지 못하여 Web에서 다시 시도

In [ ]:
import cv2
import base64
import cv2
import numpy as np
from openai import OpenAI

response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": (prompt
                    )
                }
            ]
        }
    ],
)

print(response.output_text)

### ***Chat GPT web에서 아래와 같이 프롬프트를 얻음***
- 아래 프롬프트를 초기프롬프트로 사용하기로 함

In [ ]:
initial_prompt="""### Scam indicators to look for:
The video should be considered potentially fraudulent if it contains **any** of the following characteristics:

1. **Monetary Requests:** Mentions of prepayment, fees, deposits, gift cards, or cryptocurrency transfers.
2. **Urgency or Scarcity:** Words or imagery implying “now”, “limited offer”, “only today”, “act fast”, etc.
3. **External Channel Redirection:** Mentions or visuals leading to Telegram, open chat, QR codes, or other messaging apps.
4. **Impersonation of Institutions:** Use of logos, certificates, uniforms, or documents mimicking banks, governments, media, or law enforcement.
5. **Exaggerated Profit or No-Risk Claims:** Unrealistic promises of high profit, “guaranteed returns”, or “zero risk”.
6. **Persistent Banners or Overlays:** On-screen phone numbers, repeated contact details, or fixed overlay banners.
7. **Investment/Profit/Transaction Evidence:** Screenshots or records of trading activity or fake profit statements.
8. **Visible Contact Info or Short URLs:** Display of contact numbers, wallet addresses, or shortened links (bit.ly, etc.).

---

### Output format (JSON):
After reviewing the video’s visual and textual content, respond **strictly** in the following JSON format:

{
  "is_scam": true | false,
  "confidence": float between 0.0 and 1.0,
  "risk": "low" | "mid" | "high",
  "evidence": ["concise evidence 1", "concise evidence 2", ...],
  "explanation": "A short paragraph explaining why this content is or is not a scam, referencing observed indicators."
}

---

### Guidelines:
- If **two or more scam indicators** appear, classify it as `is_scam: true`.
- Set `"risk": "high"` for strong monetary solicitation or impersonation of institutions.
- Be cautious not to classify **educational or awareness videos about scams** as scams themselves — check for warning or explanatory tone.
- Respond in **English** only.
- Do not include extra commentary or formatting outside the JSON object
"""

### ***테스트 영상 및 라벨***
- 다운로드 받은 모든 테스트 영상에 대한 라벨

In [ ]:
import os

abnormal_test_video_dir = '/home/ubuntu/video_20251104/abnormal'
normal_test_video_dir = '/home/ubuntu/video_20251104/normal'

video_paths = []
truth_labels = []

for root, dirs, files in os.walk(abnormal_test_video_dir):
    for f in files:
        file_path = os.path.join(root, f)
        video_paths.append(file_path)
        truth_labels.append('[[Abnormal]]')

for root, dirs, files in os.walk(normal_test_video_dir):
    for f in files:
        file_path = os.path.join(root, f)
        video_paths.append(file_path)
        truth_labels.append('[[Normal]]')

In [ ]:
for video_path, truth_label in zip(video_paths, truth_labels):
    print(video_path, truth_label)

### ***MiniCPM 모델 로드***
- Huggingface 코드스니펫 그대로 적용

In [ ]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()

In [ ]:
import math
import numpy as np
from PIL import Image
from moviepy.editor import VideoFileClip
import tempfile
import librosa
import ollama
import re
import traceback
import json
from json import JSONDecodeError

# -------------------------
# Extract video/audio chunks
# -------------------------
def get_video_chunk_content(video_path, flatten=True):
    video = VideoFileClip(video_path)
    print('video_duration:', video.duration)
    
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as temp_audio_file:
        temp_audio_file_path = temp_audio_file.name
        video.audio.write_audiofile(temp_audio_file_path, codec="pcm_s16le", fps=16000)
        audio_np, sr = librosa.load(temp_audio_file_path, sr=16000, mono=True)

    num_units = math.ceil(video.duration)
    contents = []
    for i in range(num_units):
        frame = video.get_frame(i+1)
        image = Image.fromarray((frame).astype(np.uint8))
        audio = audio_np[sr*i:sr*(i+1)]
        if flatten:
            contents.extend(["<unit>", image, audio])
        else:
            contents.append(["<unit>", image, audio])
    return contents

# -------------------------
# MiniCPM inference with confidence
# -------------------------
def run_minicpm(video_path, prompt, model, tokenizer):
    contents = get_video_chunk_content(video_path)
    contents.append("<unit>")
    contents.append(prompt)
    
    sys_msg = model.get_sys_prompt(mode='omni', language='en')
    msg = {"role": "user", "content": contents}
    msgs = [sys_msg, msg]
    
    res = model.chat(
        msgs=msgs,
        tokenizer=tokenizer,
        sampling=True,
        temperature=1.0,
        max_new_tokens=4096,
        omni_input=True,
        use_tts_template=False,
        generate_audio=False,
        max_slice_nums=1,
        use_image_id=False,
        return_dict=True
    )
    
    # Assume MiniCPM can return a confidence score in res["confidence"] (or we can estimate)    
    output_text = res["text"].strip()    
    
    return output_text

# -------------------------
# gpt-oss prompt correction (strong version)
# -------------------------
def correct_prompt_with_llm(old_prompt, additional_prompt):
    instruction = f"""
You are optimizing a prompt for a multimodal LLM to detect scam videos.
MiniCPM received the following prompt but made an incorrect or low-confidence prediction.

Original prompt:
\"\"\"{old_prompt}\"\"\"

{additional_prompt}
"""
    llama_response = ollama.chat(
        model="gpt-oss:20b",
        messages=[{'role': 'user', 'content': instruction}],
        options = {'temperature': 1.0}
    )
    content = llama_response['message']['content']
    return content.strip()

# -------------------------
# Automatic loop with confidence threshold
# -------------------------
def auto_loop(video_path, initial_prompt, truth_label, model, tokenizer, max_iter=5, confidence_thresh=0.9):
    prompt = initial_prompt
    prompts = []
    #for i in range(max_iter):
    output = run_minicpm(video_path, prompt, model, tokenizer)
    # print(f"[MiniCPM] iteration {i+1} output: {output}")
    print(f"[MiniCPM] output: {output}")
    
    try:
        output_dict = json.loads(output)
    except JSONDecodeError:
        additional_prompt = f"There is JSONDecodeError on MiniCPM's output. Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence with correct JSON format"
        prompt = correct_prompt_with_llm(prompt, additional_prompt)

    prompts.append(output)

    # pred = "[[Abnormal]]" if output_dict['is_scam'] else "[[Normal]]"
    # conf = output_dict['confidence']
    
    # if pred == truth_label and conf >= confidence_thresh:
    #     print("[OK] Prediction matches truth with high confidence.")
    #     break
    # else:
    #     print("[!] Prediction incorrect or low confidence, correcting prompt...")
    #     additional_prompt = f'Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence'
    #     # prompt = correct_prompt_with_llm(prompt, pred, conf, truth_label, additional_prompt)
    #     prompt = initial_prompt
    #     print(f"[!] New optimized prompt:\n{prompt}\n")
    return prompts

final_prompts = []
# for video_path, truth_label in zip(reversed(video_paths), reversed(truth_labels)):
for video_path, truth_label in zip(video_paths, truth_labels):
    data = {'video_path': video_path, 'truth_label': truth_label}
    try:        
        print(video_path)
        prompts = auto_loop(
            video_path,
            initial_prompt,
            truth_label,
            model,
            tokenizer,
            max_iter=5,
            confidence_thresh=0.9
        )
        data['prompts'] = prompts    
        final_prompts.append(data)
    except:
        data['error'] = traceback.format_exc()
        traceback.print_exc()



In [ ]:
import math
import numpy as np
from PIL import Image
from moviepy.editor import VideoFileClip
import tempfile
import librosa
import ollama
import re
import traceback
import json
from json import JSONDecodeError

# -------------------------
# Extract video/audio chunks
# -------------------------
def get_video_chunk_content(video_path, flatten=True):
    video = VideoFileClip(video_path)
    print('video_duration:', video.duration)
    
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as temp_audio_file:
        temp_audio_file_path = temp_audio_file.name
        video.audio.write_audiofile(temp_audio_file_path, codec="pcm_s16le", fps=16000)
        audio_np, sr = librosa.load(temp_audio_file_path, sr=16000, mono=True)

    num_units = math.ceil(video.duration)
    contents = []
    for i in range(num_units):
        frame = video.get_frame(i+1)
        image = Image.fromarray((frame).astype(np.uint8))
        audio = audio_np[sr*i:sr*(i+1)]
        if flatten:
            contents.extend(["<unit>", image, audio])
        else:
            contents.append(["<unit>", image, audio])
    return contents

# -------------------------
# MiniCPM inference with confidence
# -------------------------
def run_minicpm(video_path, prompt, model, tokenizer):
    contents = get_video_chunk_content(video_path)
    contents.append("<unit>")
    contents.append(prompt)
    
    sys_msg = model.get_sys_prompt(mode='omni', language='en')
    msg = {"role": "user", "content": contents}
    msgs = [sys_msg, msg]
    
    res = model.chat(
        msgs=msgs,
        tokenizer=tokenizer,
        sampling=True,
        temperature=1.0,
        max_new_tokens=4096,
        omni_input=True,
        use_tts_template=False,
        generate_audio=False,
        max_slice_nums=1,
        use_image_id=False,
        return_dict=True
    )
    
    # Assume MiniCPM can return a confidence score in res["confidence"] (or we can estimate)    
    output_text = res["text"].strip()    
    
    return output_text

# -------------------------
# gpt-oss prompt correction (strong version)
# -------------------------
def correct_prompt_with_llm(old_prompt, additional_prompt):
    instruction = f"""
You are optimizing a prompt for a multimodal LLM to detect scam videos.
MiniCPM received the following prompt but made an incorrect or low-confidence prediction.

Original prompt:
\"\"\"{old_prompt}\"\"\"

{additional_prompt}
"""
    llama_response = ollama.chat(
        model="gpt-oss:20b",
        messages=[{'role': 'user', 'content': instruction}],
        options = {'temperature': 1.0}
    )
    content = llama_response['message']['content']
    return content.strip()

# -------------------------
# Automatic loop with confidence threshold
# -------------------------
def auto_loop(video_path, initial_prompt, truth_label, model, tokenizer, max_iter=5, confidence_thresh=0.9):
    prompt = initial_prompt
    prompts = []
    #for i in range(max_iter):
    output = run_minicpm(video_path, prompt, model, tokenizer)
    # print(f"[MiniCPM] iteration {i+1} output: {output}")
    print(f"[MiniCPM] output: {output}")
    
    try:
        output_dict = json.loads(output)
    except JSONDecodeError:
        additional_prompt = f"There is JSONDecodeError on MiniCPM's output. Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence with correct JSON format"
        prompt = correct_prompt_with_llm(prompt, additional_prompt)

    prompts.append(output)

    # pred = "[[Abnormal]]" if output_dict['is_scam'] else "[[Normal]]"
    # conf = output_dict['confidence']
    
    # if pred == truth_label and conf >= confidence_thresh:
    #     print("[OK] Prediction matches truth with high confidence.")
    #     break
    # else:
    #     print("[!] Prediction incorrect or low confidence, correcting prompt...")
    #     additional_prompt = f'Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence'
    #     # prompt = correct_prompt_with_llm(prompt, pred, conf, truth_label, additional_prompt)
    #     prompt = initial_prompt
    #     print(f"[!] New optimized prompt:\n{prompt}\n")
    return prompts

final_prompts = []
for video_path, truth_label in zip(reversed(video_paths), reversed(truth_labels)):
# for video_path, truth_label in zip(video_paths, truth_labels):
    data = {'video_path': video_path, 'truth_label': truth_label}
    try:        
        print(video_path)
        prompts = auto_loop(
            video_path,
            initial_prompt,
            truth_label,
            model,
            tokenizer,
            max_iter=5,
            confidence_thresh=0.9
        )
        data['prompts'] = prompts    
        final_prompts.append(data)
    except:
        data['error'] = traceback.format_exc()
        traceback.print_exc()

